# 05 Observables

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/05_observables.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/05_observables.ipynb)

The expectation-value functions take a per-qubit Pauli string, or a weighted sum of them. This notebook shows
both, and how Qdislib groups the terms of a sum so that it runs far fewer circuits than one set per term.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qiskit]>=2.0"

In [2]:
from qiskit import QuantumCircuit
import Qdislib as qd


def example_circuit():
    """Five qubits in a line, joined by four CZ gates."""
    qc = QuantumCircuit(5)
    for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
        qc.ry(angle, qubit)
    for qubit in range(4):
        qc.cz(qubit, qubit + 1)
    for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
        qc.rx(angle, qubit)
    return qc


circuit = example_circuit()
cut = ["CZ_7"]

## Pauli strings

`observable` has one character per qubit, the first for qubit 0: `I`, `X`, `Y` or `Z`. `None` means `Z` on all qubits.
`X` and `Y` are read by rotating into the Z basis. An `I` on a cut qubit works too.

In [3]:
for observable in ["ZZZZZ", "ZIZIZ", "XZIZI", "XZZZY", "IIIII"]:
    value = qd.gate_cutting(circuit, cut, observable, shots=20_000, seed=1)
    exact = qd.analytical_solution(circuit, observable)
    print(f"{observable}  cut {value:+.3f}  exact {exact:+.3f}")

ZZZZZ  cut +0.235  exact +0.239
ZIZIZ  cut +0.252  exact +0.256
XZIZI  cut +0.621  exact +0.628


XZZZY  cut -0.309  exact -0.306
IIIII  cut +1.000  exact +1.000


## Weighted sums

A Hamiltonian is a list of `(coefficient, pauli)` pairs or a `{pauli: coefficient}` dict.

In [4]:
hamiltonian = [(0.5, "ZZIII"), (-1.0, "IIZZI"), (0.25, "ZIZII"), (0.7, "XXIII"), (2.0, "IIIII")]

value = qd.gate_cutting(circuit, cut, hamiltonian, shots=20_000, seed=1)
as_dict = qd.gate_cutting(circuit, cut, {pauli: c for c, pauli in hamiltonian}, shots=20_000, seed=1)
print(f"list {value:+.4f}  dict {as_dict:+.4f}  exact {qd.analytical_solution(circuit, hamiltonian):+.4f}")

list +1.8314  dict +1.8314  exact +1.8181


## Terms are grouped, not run one by one

Terms that are **qubit-wise commuting** (on every qubit the same Pauli or `I`) share one set of fragment circuits, and each term is read from those counts.
An identity term is a constant and runs nothing. So `ZZIII`, `IIZZI` and `ZIZII` above run once together, `XXIII` has its own run, and `IIIII` runs nothing.

The three-step list has one part per group, so its length shows it:

In [5]:
print("one Z term:     ", len(qd.gate_cutting_subcircuits(circuit, cut, "ZZIII")), "circuits")
print("the Hamiltonian:", len(qd.gate_cutting_subcircuits(circuit, cut, hamiltonian)), "circuits for", len(hamiltonian), "terms")

diagonal = [(1.0, "".join("Z" if q in (i, i + 1) else "I" for q in range(5))) for i in range(4)]
print("diagonal:       ", len(qd.gate_cutting_subcircuits(circuit, cut, diagonal)), "circuits for", len(diagonal), "terms")

one Z term:      12 circuits
the Hamiltonian: 24 circuits for 5 terms
diagonal:        12 circuits for 4 terms


A Hamiltonian that is diagonal in Z, as in QAOA, runs once whatever the number of its terms.

## The three-step form

The list recombines a sum by itself. The results of `sample_subcircuit` keep their counts, so `reconstruct(results, observable)` also reads another Pauli string in the same basis, without running anything again.

In [6]:
subs = qd.gate_cutting_subcircuits(circuit, cut, hamiltonian)
print(type(subs).__name__, len(subs), "subcircuits")
values = [qd.expectation_value(s, shots=20_000, seed=i) for i, s in enumerate(subs)]
print(f"reconstructed {subs.reconstruct(values):+.4f}")

z_subs = qd.gate_cutting_subcircuits(circuit, cut, "ZZZZZ")
results = [qd.sample_subcircuit(s, shots=20_000, seed=i) for i, s in enumerate(z_subs)]
for observable in ("ZZZZZ", "ZIIIZ", "ZZIII"):
    print(f"{observable}: {z_subs.reconstruct(results, observable):+.4f}   exact {qd.analytical_solution(circuit, observable):+.4f}")

CombinedCutSubcircuits 24 subcircuits
reconstructed +1.8316


ZZZZZ: +0.2400   exact +0.2390
ZIIIZ: +0.4080   exact +0.4055
ZZIII: +0.5996   exact +0.6019


## More terms and X, Y need more shots

The estimate is unbiased, but its variance depends on the observable as well as on `gamma`. `error=True` (notebook 11) shows it for each one.

In [7]:
for observable in ["ZZZZZ", "XZZZY", "IIIII", "XIIII"]:
    print(observable, qd.gate_cutting(circuit, cut, observable, shots=4000, seed=1, error=True))

ZZZZZ 0.241199 ± 0.01
XZZZY -0.312364 ± 0.011
IIIII 1.0


XIIII 0.636476 ± 0.011


Next: **06 Sampling**.